# 95 — Memory Compaction
## MemGPT-Style 3-Tier Memory: Keep Agents Sharp Across Thousands of Turns
⏱ ~45 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/95-memory-compaction/memory_compaction_workbook.ipynb)

Every LLM has a fixed context window. In a long-running agent — a trading assistant, a support bot, a personal AI — conversations accumulate hundreds or thousands of turns. Naively stuffing every turn into the context hits the token limit fast, and truncating old turns means the agent forgets crucial user facts.

**Memory compaction** solves this by managing memory across three tiers, each with a different fidelity/cost trade-off:
- **Hot** — verbatim recent turns, always in context (high fidelity, limited slots)
- **Warm** — LLM-generated summaries of compacted hot turns, always in context (compressed, but present)
- **Cold** — oldest summary blocks, archived and never auto-loaded (off-context storage)

This pattern was introduced by Packer et al. in the MemGPT paper (2023). We implement it as a 4-node LangGraph: `load_context → respond → save_turn → compact`.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — Why context limits are a hard problem |
| 2 | **Setup** — Install deps, load API key |
| 3 | **Tools** — Importance scoring and context assembly |
| 4 | **State** — The `MemState` TypedDict |
| 5 | **Graph Nodes** — load_context, respond, save_turn, compact |
| 6 | **Workflow** — Assembling the 4-node LangGraph |
| 7 | **Demo** — Running 12 turns and watching the tiers shift |
| 8 | **Tier Inspection** — Examining what lives where after compaction |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- `langchain-openai`, `langgraph`, `python-dotenv`

### Key References
> Packer et al. (2023). *MemGPT: Towards LLMs as Operating Systems.* arXiv:2310.08560  
> https://arxiv.org/abs/2310.08560
>
> LangGraph Documentation: https://langchain-ai.github.io/langgraph/

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "langchain-openai", "langgraph", "python-dotenv", "typing_extensions"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local — skipping install. Ensure deps are in your venv.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
if not (bool(key) and key.startswith('sk-')):
    raise RuntimeError("OPENAI_API_KEY is required for the live memory-compaction cells.")
print("API key ready: True")

---
## Part 1 — Why Context Limits Are a Hard Problem

### The Forgetting Problem

An LLM processes every request fresh — it has no persistent memory between API calls. State lives entirely in the **context window** you pass on each call. This creates a hard ceiling:

| Model | Context window |
|-------|---------------|
| gpt-4o-mini | 128 k tokens |
| gpt-4o | 128 k tokens |
| claude-3-7-sonnet | 200 k tokens |

At ~500 tokens per turn (user + assistant), a 128 k window fills in **~256 turns**. That sounds large, but:
- A customer support bot fielding 10 msg/minute fills it in 25 minutes.
- A personal assistant accumulating months of interactions overflows immediately.

### Naive Approaches and Their Flaws

```
Approach            Flaw
─────────────────── ────────────────────────────────────────────
Sliding window      Drops old turns verbatim → agent forgets facts
Truncate oldest     Same as sliding window
Keep everything     Hits token limit; slow + expensive inference
Embed + retrieve    Loses turn order; retrieval adds latency + complexity
```

### The MemGPT Insight

MemGPT (Packer et al. 2023) frames the LLM as an **operating system** managing a memory hierarchy — inspired by how CPUs juggle registers, RAM, and disk:

```
  HOT   ┌─────────────────────┐  verbatim turns   — always in context
 (fast) │  turn N-5 … turn N  │  high fidelity    — limited slots
        └─────────────────────┘
               │ compaction (LLM summarizes)
               ▼
  WARM  ┌─────────────────────┐  LLM summaries    — always in context
 (med)  │  summary 1 … K      │  compressed        — moderate slots
        └─────────────────────┘
               │ archive (overflow)
               ▼
  COLD  ┌─────────────────────┐  oldest summaries — never auto-loaded
 (disk) │  block 1 … M        │  off-context       — unlimited storage
        └─────────────────────┘
```

The critical design choices:
1. **Which turns to compact?** — use an **importance score** so user facts ("remember", "never", "always") stay verbatim longer.
2. **What to preserve in summaries?** — the LLM prompt instructs: *"preserve every user fact."*
3. **When to archive warm → cold?** — when warm overflows, oldest blocks go to cold.

---
## Part 2 — Setup: Constants and Demo Turns

We define two tier-size constants that control when compaction fires, and a set of 12 demo conversation turns that simulate a real multi-session trading assistant conversation.

Notice how several turns include explicit memory cues (`"Remember that..."`, `"Don't forget..."`, `"never"`). The importance scorer gives these turns a boost so they survive compaction longer — staying verbatim in the hot tier instead of being summarized away.

In [ ]:
# Tier size limits — tweak these to experiment
HOT_TIER_MAX = 6   # verbatim turns before compaction fires
WARM_TIER_MAX = 4  # summary blocks before oldest archive to cold

# 12 demo turns — a realistic trading assistant conversation
DEMO_TURNS = [
    "My name is Alex and I'm building a trading bot.",
    "Remember that I only trade ETH, never BTC.",
    "What's the difference between a limit order and a market order?",
    "I'm using Binance as my exchange.",
    "Remember that my risk tolerance is low — max 2% per trade.",
    "Explain slippage to me.",
    "What timeframes work best for ETH?",
    "Don't forget I'm based in Canada so tax rules matter.",
    "How do I calculate position size?",
    "What is the Kelly criterion?",
    "I prefer 4H and daily charts.",
    "How do stop losses interact with volatility?",
]

print(f"HOT_TIER_MAX  = {HOT_TIER_MAX}  (compaction fires when hot > {HOT_TIER_MAX} turns)")
print(f"WARM_TIER_MAX = {WARM_TIER_MAX}  (archive fires when warm > {WARM_TIER_MAX} blocks)")
print(f"Total demo turns: {len(DEMO_TURNS)}")

# Identify which turns have explicit memory cues
cues = {"remember", "don't forget", "never", "always", "important"}
flagged = [(i+1, t[:60]) for i, t in enumerate(DEMO_TURNS)
           if any(c in t.lower() for c in cues)]
print("\nTurns with explicit memory cues (will get importance boost):")
for n, t in flagged:
    print(f"  Turn {n:02d}: {t}")

---
## Part 3 — Tools: Importance Scoring and Context Assembly

### Importance Score

When hot overflows, which turns should be compacted? We need a score — higher score = more important = stays verbatim longer.

The score combines two signals:
- **Recency** (`0.0` – `1.0`): starts at `1.0` when a turn is added, decays by 10% each step. Older turns become less important.
- **Keyword boost** (`+0.4`): if the turn contains `"remember"`, `"don't forget"`, `"never"`, `"always"`, or `"important"`, it gets a 0.4 boost. Capped at `1.0`.

```
importance = min(1.0, recency + boost)

Example:
  turn: "Remember that I only trade ETH, never BTC."
  recency = 0.7 (3 steps old, decayed 3x)
  boost   = 0.4 (contains "remember" and "never")
  score   = min(1.0, 0.7 + 0.4) = 1.0  ← stays verbatim

  turn: "What's the difference between a limit order and a market order?"
  recency = 0.7 (3 steps old)
  boost   = 0.0 (no cues)
  score   = 0.7  ← candidate for compaction
```

### Context Assembly

The `build_context` function stitches warm summaries (oldest first) and hot turns into a single string the LLM uses as its system context. This gives the model both the high-level narrative (warm) and the precise recent dialogue (hot).

In [ ]:
def importance_score(turn: dict) -> float:
    """Recency x (1 + boost) — explicit memory cues ('remember', 'never') lift score."""
    recency = turn.get("recency", 0.5)
    content = turn.get("content", "").lower()
    cues = {"remember", "don't forget", "never", "always", "important"}
    boost = 0.4 if any(c in content for c in cues) else 0.0
    return min(1.0, recency + boost)


def build_context(hot: list, warm: list) -> str:
    """Concatenate warm summaries then recent verbatim turns into a single context block."""
    lines = []
    if warm:
        lines.append("=== Summarized history ===")
        for block in warm:
            lines.append(block["summary"])
    if hot:
        lines.append("=== Recent turns ===")
        for t in hot:
            lines.append(f"{t['role']}: {t['content']}")
    return "\n".join(lines)


# --- Demonstrate importance_score ---
print("Importance score examples:")
examples = [
    {"content": "Remember that I only trade ETH, never BTC.", "recency": 0.7},
    {"content": "What's the difference between a limit order and a market order?", "recency": 0.7},
    {"content": "What timeframes work best for ETH?", "recency": 0.4},
    {"content": "My risk tolerance is important — never exceed 2% per trade.", "recency": 0.3},
]
for ex in examples:
    score = importance_score(ex)
    print(f"  score={score:.2f}  recency={ex['recency']}  '{ex['content'][:55]}'")

In [ ]:
# --- Demonstrate build_context ---
sample_hot = [
    {"role": "user",      "content": "What timeframes work best for ETH?",  "recency": 0.8},
    {"role": "assistant", "content": "4H and daily are popular for ETH...",  "recency": 0.7},
]
sample_warm = [
    {"summary": "Alex is building a trading bot on Binance, trades only ETH, risk tolerance low at max 2% per trade."}
]

ctx = build_context(sample_hot, sample_warm)
print("build_context output:")
print("-" * 60)
print(ctx)
print("-" * 60)
print(f"\nTotal context length: {len(ctx)} chars")

---
## Part 4 — State: The MemState TypedDict

LangGraph requires a typed state dict that flows through every node. Every node reads from it and returns a partial update — LangGraph merges the update back into state.

```
MemState
├── query      str        — current user message
├── response   str        — LLM reply for this turn
├── hot_tier   list[dict] — verbatim recent turns   (always in context)
├── warm_tier  list[dict] — LLM summaries of compacted hot turns
├── cold_tier  list[dict] — oldest summaries, archived (off-context)
└── context    str        — assembled context string for this turn
```

Each turn dict in `hot_tier` has three fields:
```python
{"role": "user" | "assistant", "content": "...", "recency": 0.0–1.0}
```

Each block in `warm_tier` and `cold_tier` has one field:
```python
{"summary": "..."}
```

In [ ]:
from typing_extensions import TypedDict


class MemState(TypedDict):
    query: str
    response: str
    hot_tier: list   # verbatim recent turns — always in context
    warm_tier: list  # LLM summaries of compacted hot turns
    cold_tier: list  # oldest summaries; never auto-loaded
    context: str


# Initial empty state
initial_state: MemState = {
    "query": "",
    "response": "",
    "hot_tier": [],
    "warm_tier": [],
    "cold_tier": [],
    "context": "",
}

print("MemState fields:", list(initial_state.keys()))
print("All tiers empty at start:",
      len(initial_state["hot_tier"]) == 0 and
      len(initial_state["warm_tier"]) == 0 and
      len(initial_state["cold_tier"]) == 0)

---
## Part 5 — Graph Nodes: The Four Functions

The workflow has exactly four nodes. Each is a pure function `(state) → dict` — it reads state, does one thing, and returns only the fields it changed.

```
START
  │
  ▼
load_context   ── assembles context string from hot + warm tiers
  │
  ▼
respond        ── calls LLM with context + user query
  │
  ▼
save_turn      ── appends exchange to hot_tier; decays recency
  │
  ▼
compact        ── if hot overflows: summarize low-importance turns → warm
  │              if warm overflows: archive oldest blocks → cold
  ▼
END
```

### Node 1: `load_context`
Simple — just calls `build_context` and stores the result.

### Node 2: `respond`
Calls the LLM. The context string becomes the system prompt, giving the model its memory.

### Node 3: `save_turn`
Adds both sides of the exchange (user + assistant) to `hot_tier`. Before appending, it decays every existing turn's `recency` by 10% — simulating time passing.

### Node 4: `compact`
The heart of the system:
1. If `len(hot) <= HOT_TIER_MAX` → no-op, return `{}`
2. Sort hot turns by importance score (ascending) — lowest first
3. Select the `n = len(hot) - HOT_TIER_MAX // 2` lowest-importance turns to drop
4. Ask the LLM to summarize those turns → new warm block
5. If `len(warm) > WARM_TIER_MAX` → move oldest warm blocks to cold

In [ ]:
from langchain_openai import ChatOpenAI

_llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)


def load_context(state: MemState) -> dict:
    return {"context": build_context(state["hot_tier"], state["warm_tier"])}


def respond(state: MemState) -> dict:
    system = "You are a helpful assistant with access to the conversation history below.\n\n" + state["context"]
    reply = _llm.invoke([
        {"role": "system", "content": system},
        {"role": "user",   "content": state["query"]},
    ])
    return {"response": reply.content}


def save_turn(state: MemState) -> dict:
    """Append exchange; decay recency of existing hot turns by 10% each step."""
    decayed = [{**t, "recency": t.get("recency", 1.0) * 0.9} for t in state["hot_tier"]]
    decayed.append({"role": "user",      "content": state["query"],    "recency": 1.0})
    decayed.append({"role": "assistant", "content": state["response"], "recency": 0.9})
    return {"hot_tier": decayed}


def compact(state: MemState) -> dict:
    """Lowest-importance hot turns -> warm summary; oldest warm blocks -> cold archive."""
    hot, warm, cold = list(state["hot_tier"]), list(state["warm_tier"]), list(state["cold_tier"])
    if len(hot) <= HOT_TIER_MAX:
        return {}

    n = len(hot) - HOT_TIER_MAX // 2
    order = sorted(range(len(hot)), key=lambda i: importance_score(hot[i]))
    drop = set(order[:n])
    text = "\n".join(f"{hot[i]['role']}: {hot[i]['content']}" for i in sorted(drop))
    hot = [t for i, t in enumerate(hot) if i not in drop]

    summary = _llm.invoke([
        {"role": "system", "content": "Summarize in 2-3 sentences, preserving every user fact."},
        {"role": "user",   "content": text},
    ]).content
    warm.append({"summary": summary})

    if len(warm) > WARM_TIER_MAX:
        overflow = len(warm) - WARM_TIER_MAX
        cold.extend(warm[:overflow])
        warm = warm[overflow:]

    return {"hot_tier": hot, "warm_tier": warm, "cold_tier": cold}


print("All four node functions defined.")
print("LLM:", _llm.model_name)

---
## Part 6 — Assembling the Workflow

LangGraph's `StateGraph` connects nodes with directed edges. The graph compiles to an `app` object with an `.invoke()` method — the same interface as a LangChain runnable.

```python
StateGraph(MemState)
  add_node("load_context", load_context)
  add_node("respond",      respond)
  add_node("save_turn",    save_turn)
  add_node("compact",      compact)
  add_edge(START → "load_context" → "respond" → "save_turn" → "compact" → END)
  compile()
```

The resulting graph is a **pure linear chain** — no conditionals, no branching. The `compact` node handles its own no-op case by returning `{}` when the hot tier is within limits.

In [ ]:
from langgraph.graph import StateGraph, START, END


def create_workflow():
    g = StateGraph(MemState)
    for name, fn in [
        ("load_context", load_context),
        ("respond",      respond),
        ("save_turn",    save_turn),
        ("compact",      compact),
    ]:
        g.add_node(name, fn)

    g.add_edge(START, "load_context")
    g.add_edge("load_context", "respond")
    g.add_edge("respond", "save_turn")
    g.add_edge("save_turn", "compact")
    g.add_edge("compact", END)

    return g.compile()


app = create_workflow()
print("Workflow compiled successfully.")
print("Nodes:", list(app.graph.nodes))

---
## Part 7 — Running the Demo: 12 Turns of Tier Evolution

We run all 12 demo turns sequentially through the workflow, carrying state forward between turns. After each turn, we print the tier sizes so you can watch compaction happen in real time.

**What to watch for:**
- Hot tier grows until it exceeds `HOT_TIER_MAX = 6`
- Compaction fires → warm tier gains a new summary block, hot tier shrinks
- After enough compactions, warm tier may overflow → cold tier gains blocks
- The `← compacted` flag appears on turns where compaction occurred

In [ ]:
print("=== 95 · Memory Compaction ===")
print("hot (verbatim) -> warm (LLM summaries) -> cold (archive)\n")

state: dict = {
    "query": "", "response": "",
    "hot_tier": [], "warm_tier": [], "cold_tier": [], "context": "",
}

for i, turn in enumerate(DEMO_TURNS, start=1):
    state = app.invoke({**state, "query": turn})
    h, w, c = len(state["hot_tier"]), len(state["warm_tier"]), len(state["cold_tier"])
    flag = " <- compacted" if (w + c) > 0 else ""

    print(f"[{i:02d}] {turn[:70]}")
    print(f"      {state['response'][:90]}...")
    print(f"      hot={h}  warm={w}  cold={c}{flag}\n")

In [ ]:
# Final tier summary
print("Final tiers after all 12 turns:")
print(f"  Hot  {len(state['hot_tier'])} turns   — verbatim, always in context")
print(f"  Warm {len(state['warm_tier'])} blocks  — LLM summaries, always in context")
print(f"  Cold {len(state['cold_tier'])} blocks  — archived, never auto-loaded")

---
## Part 8 — Tier Inspection: What Lives Where?

After running all turns, we can inspect the contents of each tier to understand what the compaction preserved, compressed, and archived. This is a key diagnostic for real deployments — you want to verify that critical user facts survive compaction.

In [ ]:
print("=" * 60)
print("HOT TIER — verbatim turns currently in context")
print("=" * 60)
for i, t in enumerate(state["hot_tier"]):
    score = importance_score(t)
    print(f"  [{i}] role={t['role']:<10} recency={t['recency']:.2f}  score={score:.2f}")
    print(f"       {t['content'][:80]}")

In [ ]:
print("=" * 60)
print("WARM TIER — LLM summaries currently in context")
print("=" * 60)
if state["warm_tier"]:
    for i, block in enumerate(state["warm_tier"]):
        print(f"  [Block {i}]: {block['summary']}")
        print()
else:
    print("  (empty — no compaction occurred yet)")

In [ ]:
print("=" * 60)
print("COLD TIER — archived summaries, off-context")
print("=" * 60)
if state["cold_tier"]:
    for i, block in enumerate(state["cold_tier"]):
        print(f"  [Archive {i}]: {block['summary']}")
        print()
else:
    print("  (empty — warm tier has not overflowed yet)")

---
## Part 9 — Inspecting the Context Window

The `context` field in state shows exactly what the model receives as its system prompt history for any given turn. This is the assembled output of `build_context(hot, warm)` — a key diagnostic for understanding what the agent knows.

In [ ]:
print("Context passed to LLM on the final turn:")
print("=" * 60)
print(state["context"] if state["context"] else "(empty — no history yet)")
print("=" * 60)
print(f"\nContext length: {len(state['context'])} characters")

# Estimate approximate token cost (rough: 4 chars per token)
approx_tokens = len(state["context"]) // 4
print(f"Approx tokens in context: ~{approx_tokens} (rough estimate, 4 chars/token)")

---
## Part 10 — Recency Decay: Visualizing Score Over Time

The recency decay is a 10% exponential decay per turn. Combined with the keyword boost, it determines which turns get compacted first. Let's plot how scores evolve for a boosted vs non-boosted turn.

In [ ]:
# Simulate recency decay over 12 steps
steps = list(range(0, 13))

# Turn added at step 0 with recency=1.0
no_boost   = [min(1.0, (1.0 * (0.9 ** s)))        for s in steps]  # plain turn
with_boost = [min(1.0, (1.0 * (0.9 ** s)) + 0.4)  for s in steps]  # cue turn

print("Step | Recency | Score (no boost) | Score (with boost) | Compaction candidate?")
print("-" * 75)
for s in steps:
    r = 1.0 * (0.9 ** s)
    nb = min(1.0, r)
    wb = min(1.0, r + 0.4)
    candidate_no  = "YES" if nb < 0.6 else "   "
    candidate_yes = "YES" if wb < 0.6 else "   "
    print(f"  {s:2d}  | {r:6.3f}  |      {nb:.3f}  {candidate_no}    |      {wb:.3f}  {candidate_yes}        |")

---
## Part 11 — Compact Node Deep Dive

Let's trace through the compact algorithm step by step on a synthetic hot tier to make the selection logic crystal clear.

**Algorithm:**
1. If `len(hot) <= HOT_TIER_MAX` → return `{}` (no-op)
2. `n = len(hot) - HOT_TIER_MAX // 2` — how many to drop
3. Sort turn indices by importance score (ascending) — lowest importance first
4. `drop = set(order[:n])` — select the n lowest-importance turns
5. Build text from dropped turns → LLM summarizes → append to warm
6. Remove dropped turns from hot
7. If `len(warm) > WARM_TIER_MAX` → overflow oldest blocks to cold

In [ ]:
# Dry-run compact selection logic (no LLM call — just show which turns get dropped)
synthetic_hot = [
    {"role": "user",      "content": "My name is Alex.",                   "recency": 0.53},
    {"role": "assistant", "content": "Nice to meet you, Alex!",            "recency": 0.47},
    {"role": "user",      "content": "Remember: I only trade ETH, never BTC.", "recency": 0.43},
    {"role": "assistant", "content": "Got it — ETH only.",                 "recency": 0.38},
    {"role": "user",      "content": "What is a limit order?",             "recency": 0.34},
    {"role": "assistant", "content": "A limit order executes at a set price.", "recency": 0.31},
    {"role": "user",      "content": "I'm on Binance.",                    "recency": 1.0},
    {"role": "assistant", "content": "Binance noted!",                     "recency": 0.9},
]

hot = synthetic_hot
n = len(hot) - HOT_TIER_MAX // 2
order = sorted(range(len(hot)), key=lambda i: importance_score(hot[i]))
drop = set(order[:n])

print(f"Hot tier size: {len(hot)}, HOT_TIER_MAX: {HOT_TIER_MAX}")
print(f"HOT_TIER_MAX // 2 = {HOT_TIER_MAX // 2}  (target remaining after compaction)")
print(f"n = len(hot) - HOT_TIER_MAX // 2 = {len(hot)} - {HOT_TIER_MAX // 2} = {n} turns to drop\n")

print("Turn analysis (sorted by importance score):")
print(f"{'idx':<4} {'score':<7} {'status':<10} content")
print("-" * 70)
for rank, idx in enumerate(order):
    t = hot[idx]
    score = importance_score(t)
    status = "DROP" if idx in drop else "KEEP"
    print(f"{idx:<4} {score:<7.3f} {status:<10} {t['content'][:50]}")

print(f"\nTurns kept in hot after compaction: {[i for i in range(len(hot)) if i not in drop]}")
print("Turns to be summarized into warm:")
for i in sorted(drop):
    print(f"  [{i}] {hot[i]['role']}: {hot[i]['content'][:60]}")

---
## Part 12 — Single-Turn Sanity Check

Before running the full demo, it's useful to test a single turn in isolation to verify the workflow is wired correctly and the LLM is responding. This is a good debugging pattern for any LangGraph workflow.

In [ ]:
# Single-turn test
test_state: dict = {
    "query": "What is memory compaction in the context of AI agents?",
    "response": "",
    "hot_tier": [],
    "warm_tier": [],
    "cold_tier": [],
    "context": "",
}

result = app.invoke(test_state)

print("Query:", test_state["query"])
print()
print("Response:", result["response"][:300])
print()
print(f"Hot tier after first turn: {len(result['hot_tier'])} items")
print(f"Warm tier: {len(result['warm_tier'])}  Cold tier: {len(result['cold_tier'])}")

---
## Part 13 — Design Trade-offs and Extensions

### What this implementation does well
- **Simple** — 4 nodes, 2 helper functions, no external vector DB
- **Importance-aware** — explicit user facts survive compaction longer
- **Three-tier hierarchy** — mirrors MemGPT's operating system analogy
- **Stateless nodes** — easy to test, serialize, or swap

### What it doesn't do (extensions for production)

| Extension | Why it matters |
|-----------|----------------|
| **Cold retrieval** | Load cold blocks on-demand when query is relevant — requires embedding + similarity search |
| **Async compaction** | Run compact in background while respond is streaming — reduces latency |
| **User-controlled memory** | `"forget that"` or `"remember this forever"` commands — needs a memory management node |
| **Persistent state** | Serialize `hot_tier`/`warm_tier`/`cold_tier` to a database between sessions |
| **Semantic importance** | Use embeddings instead of keyword matching for importance scoring |
| **Conditional compaction** | Branch on `compact_needed` instead of always running the compact node |
| **Token budget** | Count actual tokens instead of turn count — avoids variable-length turn surprises |

### Key configuration levers
```python
HOT_TIER_MAX = 6   # Higher = more verbatim context, but larger prompts
WARM_TIER_MAX = 4  # Higher = more summaries in context, lower = more cold archival
recency_decay = 0.9  # Lower = old turns die faster; 0.5 = aggressive
boost = 0.4        # How much keyword cues protect turns from compaction
```

---
## Exercises

### Exercise 1 — Tune the Hot Tier Size

Change `HOT_TIER_MAX` to `4` and re-run the full demo. Observe:
- Does compaction fire earlier?
- Do more warm blocks accumulate?
- Does the agent still remember Alex's name and ETH-only constraint?

**Hint:** Set `HOT_TIER_MAX = 4` at the top of the constants cell, rebuild the workflow, and re-run the demo loop.

---

### Exercise 2 — Stronger Keyword Boost

Modify `importance_score` to boost turns containing `"remember"` or `"never"` by `0.7` instead of `0.4`. Then run the demo and check the compact dry-run to see if those turns are now always kept verbatim.

**Hint:** Change `boost = 0.4` to `boost = 0.7` in the `importance_score` function.

---

### Exercise 3 — Token-Based Hot Tier

Instead of counting turns (`HOT_TIER_MAX = 6`), implement a token-budget version: compact when the total character length of all hot turns exceeds `TOKEN_BUDGET_CHARS = 1500`. This is more realistic for production systems where turns vary wildly in length.

**Hint:** Modify the `compact` function's first check from `len(hot) <= HOT_TIER_MAX` to a character-count check. You'll also need to decide how many turns to drop when budget is exceeded.

In [ ]:
# === Answer Key ===

# --- Exercise 1: HOT_TIER_MAX = 4 ---
HOT_TIER_MAX_EX1 = 4
WARM_TIER_MAX_EX1 = 4

# Rebuild workflow with smaller hot tier
def compact_ex1(state: MemState) -> dict:
    hot, warm, cold = list(state["hot_tier"]), list(state["warm_tier"]), list(state["cold_tier"])
    if len(hot) <= HOT_TIER_MAX_EX1:
        return {}
    n = len(hot) - HOT_TIER_MAX_EX1 // 2
    order = sorted(range(len(hot)), key=lambda i: importance_score(hot[i]))
    drop = set(order[:n])
    text = "\n".join(f"{hot[i]['role']}: {hot[i]['content']}" for i in sorted(drop))
    hot = [t for i, t in enumerate(hot) if i not in drop]
    summary = _llm.invoke([
        {"role": "system", "content": "Summarize in 2-3 sentences, preserving every user fact."},
        {"role": "user",   "content": text},
    ]).content
    warm.append({"summary": summary})
    if len(warm) > WARM_TIER_MAX_EX1:
        overflow = len(warm) - WARM_TIER_MAX_EX1
        cold.extend(warm[:overflow])
        warm = warm[overflow:]
    return {"hot_tier": hot, "warm_tier": warm, "cold_tier": cold}

g_ex1 = StateGraph(MemState)
for name, fn in [("load_context", load_context), ("respond", respond),
                 ("save_turn", save_turn), ("compact", compact_ex1)]:
    g_ex1.add_node(name, fn)
g_ex1.add_edge(START, "load_context")
g_ex1.add_edge("load_context", "respond")
g_ex1.add_edge("respond", "save_turn")
g_ex1.add_edge("save_turn", "compact")
g_ex1.add_edge("compact", END)
app_ex1 = g_ex1.compile()

print("Exercise 1: HOT_TIER_MAX=4 workflow")
state_ex1: dict = {"query": "", "response": "", "hot_tier": [], "warm_tier": [], "cold_tier": [], "context": ""}
for i, turn in enumerate(DEMO_TURNS[:6], start=1):  # first 6 turns only for brevity
    state_ex1 = app_ex1.invoke({**state_ex1, "query": turn})
    h, w, c = len(state_ex1["hot_tier"]), len(state_ex1["warm_tier"]), len(state_ex1["cold_tier"])
    print(f"  [{i:02d}] hot={h}  warm={w}  cold={c}")
print()

In [ ]:
# --- Exercise 2: Stronger keyword boost (0.7) ---
def importance_score_ex2(turn: dict) -> float:
    """Same as importance_score but with a stronger 0.7 boost for memory cues."""
    recency = turn.get("recency", 0.5)
    content = turn.get("content", "").lower()
    cues = {"remember", "don't forget", "never", "always", "important"}
    boost = 0.7 if any(c in content for c in cues) else 0.0  # boosted from 0.4 → 0.7
    return min(1.0, recency + boost)

print("Exercise 2: Stronger boost (0.7) vs original (0.4)")
cue_turn    = {"content": "Remember that I only trade ETH, never BTC.", "recency": 0.3}
plain_turn  = {"content": "What is a limit order?", "recency": 0.3}

print(f"  Cue turn   — original: {importance_score(cue_turn):.2f}   boosted: {importance_score_ex2(cue_turn):.2f}")
print(f"  Plain turn — original: {importance_score(plain_turn):.2f}   boosted: {importance_score_ex2(plain_turn):.2f}")
print("  With boost=0.7, cue turn is always 1.0 even at recency=0.3 — never a compaction candidate.")

In [ ]:
# --- Exercise 3: Token-budget hot tier ---
TOKEN_BUDGET_CHARS = 1500  # compact when total hot-tier char length exceeds this

def compact_token_budget(state: MemState) -> dict:
    """Compact based on character budget instead of turn count."""
    hot, warm, cold = list(state["hot_tier"]), list(state["warm_tier"]), list(state["cold_tier"])
    total_chars = sum(len(t.get("content", "")) for t in hot)
    if total_chars <= TOKEN_BUDGET_CHARS:
        return {}

    # Drop turns until we're back under 70% of the budget
    target = int(TOKEN_BUDGET_CHARS * 0.7)
    order = sorted(range(len(hot)), key=lambda i: importance_score(hot[i]))
    drop = set()
    chars_dropped = 0
    for idx in order:
        if total_chars - chars_dropped <= target:
            break
        chars_dropped += len(hot[idx].get("content", ""))
        drop.add(idx)

    if not drop:
        return {}

    text = "\n".join(f"{hot[i]['role']}: {hot[i]['content']}" for i in sorted(drop))
    hot = [t for i, t in enumerate(hot) if i not in drop]
    summary = _llm.invoke([
        {"role": "system", "content": "Summarize in 2-3 sentences, preserving every user fact."},
        {"role": "user",   "content": text},
    ]).content
    warm.append({"summary": summary})
    if len(warm) > WARM_TIER_MAX:
        overflow = len(warm) - WARM_TIER_MAX
        cold.extend(warm[:overflow])
        warm = warm[overflow:]
    return {"hot_tier": hot, "warm_tier": warm, "cold_tier": cold}


g_ex3 = StateGraph(MemState)
for name, fn in [("load_context", load_context), ("respond", respond),
                 ("save_turn", save_turn), ("compact", compact_token_budget)]:
    g_ex3.add_node(name, fn)
g_ex3.add_edge(START, "load_context")
g_ex3.add_edge("load_context", "respond")
g_ex3.add_edge("respond", "save_turn")
g_ex3.add_edge("save_turn", "compact")
g_ex3.add_edge("compact", END)
app_ex3 = g_ex3.compile()

print(f"Exercise 3: Token-budget compact (budget={TOKEN_BUDGET_CHARS} chars)")
state_ex3: dict = {"query": "", "response": "", "hot_tier": [], "warm_tier": [], "cold_tier": [], "context": ""}
for i, turn in enumerate(DEMO_TURNS[:6], start=1):
    state_ex3 = app_ex3.invoke({**state_ex3, "query": turn})
    hot_chars = sum(len(t.get("content", "")) for t in state_ex3["hot_tier"])
    h, w, c = len(state_ex3["hot_tier"]), len(state_ex3["warm_tier"]), len(state_ex3["cold_tier"])
    print(f"  [{i:02d}] hot={h} ({hot_chars} chars)  warm={w}  cold={c}")

---
## Workshop Complete

You built a MemGPT-style 3-tier memory compaction system from scratch:

- **Hot tier** — verbatim recent turns with recency decay
- **Warm tier** — LLM-generated summaries of compacted turns
- **Cold tier** — archived blocks, off-context
- **Importance scoring** — keyword boost protects explicit user facts
- **4-node LangGraph** — `load_context → respond → save_turn → compact`

This pattern enables agents to run indefinitely without hitting context limits, while preserving the facts that matter most.

---

**Next:** Example 96 — Extended Thinking  
Explore how chain-of-thought and extended reasoning tokens improve complex multi-step agent decisions.

---

**Reference:**  
Packer et al. (2023). *MemGPT: Towards LLMs as Operating Systems.* [arXiv:2310.08560](https://arxiv.org/abs/2310.08560)